# Chapter 1 — Applied Examples
## ARIMA · Cointegration & ECM · GARCH

**Macroeconometrics** | Alessia Paccagnini | Companion Notebook

---

This notebook contains three self-contained applied examples that bridge the theory of Chapter 1 to practice. Each example follows a consistent structure: data generation → estimation → diagnostics → interpretation.

| Example | Topic | Key methods |
|---|---|---|
| **1** | ARIMA Model Selection | Box-Jenkins (Identify → Estimate → Check), ADF/KPSS, Ljung-Box |
| **2** | Spurious Regression, Cointegration & ECM | Engle-Granger two-step, ADF on residuals, Error Correction |
| **3** | GARCH Volatility Modelling | ARCH-LM test, GARCH(1,1), GJR-GARCH, EGARCH, half-life |

**Design conventions used throughout:**
- 🎛️ cells mark tunable parameters — change and re-run freely
- Every code block has a **header comment** explaining *why* this step is needed
- **Inline comments** on every non-trivial line explain *what* it does
- Every figure is followed by a **"What to look for"** note
- Figures are saved as `.png` and `.pdf` for use in LaTeX/Overleaf

---

In [ ]:
# ── Shared imports (run this cell first) ─────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings, os
warnings.filterwarnings('ignore')

# statsmodels — used in Examples 1 and 2
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.regression.linear_model import OLS
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# arch — used in Example 3 (install: pip install arch)
from arch import arch_model

# ── Output directory ──────────────────────────────────────────────────────────
OUT_DIR = '.'
os.makedirs(OUT_DIR, exist_ok=True)

# ── Shared plot style ─────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130,
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'legend.fontsize': 10,
})

print('All packages loaded successfully.')

---
# Example 1 — ARIMA Model Selection
### Box-Jenkins Three-Stage Methodology
*Textbook reference: Sections 1.5–1.6*

## Background

The **Box-Jenkins methodology** (1970) remains the standard workflow for fitting univariate ARIMA models. It proceeds in three stages:

1. **Identification**: Is the series stationary? If not, how many times must we difference it ($d$)? What are the likely AR order $p$ and MA order $q$ from the ACF/PACF?
2. **Estimation**: Fit candidate ARIMA($p,d,q$) models by maximum likelihood and compare using information criteria (AIC, BIC).
3. **Diagnostic checking**: Are the residuals white noise? If not, the model is misspecified and we return to Stage 1.

The general ARIMA($p,d,q$) model writes:

$$\phi(L)(1-L)^d y_t = c + \theta(L)\varepsilon_t, \quad \varepsilon_t \sim WN(0,\sigma^2)$$

where $\phi(L) = 1 - \phi_1 L - \cdots - \phi_p L^p$ and $\theta(L) = 1 + \theta_1 L + \cdots + \theta_q L^q$.

**In this example** we simulate quarterly GDP growth as an ARMA(2,1) process (so the true model is known), then pretend we do not know this and apply Box-Jenkins to recover it.

In [ ]:
# 🎛️ EXAMPLE 1 PARAMETERS
N1      = 200     # sample size (quarterly obs.)
SEED1   = 42      # reproducibility seed
# True DGP: ARMA(2,1) — the model selection should recover this
PHI1    =  0.4   # AR(1) coefficient
PHI2    = -0.2   # AR(2) coefficient
THETA1  =  0.3   # MA(1) coefficient
CONST1  =  0.5   # intercept (mean GDP growth ≈ 2% annualised)

# Models to compare in Stage 2
CANDIDATE_ORDERS = [(1,0,0), (2,0,0), (1,0,1), (2,0,1), (2,0,2)]

print(f'True DGP: ARMA(2,1)  →  y_t = {CONST1} + {PHI1}·y_(t-1) + ({PHI2})·y_(t-2) + ε_t + {THETA1}·ε_(t-1)')

## Stage 0 — Data simulation

We simulate GDP growth as an ARMA(2,1). In a real application you would load your data here (e.g. from FRED). The simulation lets us verify later that Box-Jenkins recovers the correct model order.

In [ ]:
# ── Stage 0: Simulate data ────────────────────────────────────────────────────
np.random.seed(SEED1)

eps = np.random.normal(0, 1, N1)   # white noise shocks
y1  = np.zeros(N1)

# Recursive formula: ARMA(2,1)
# Needs at least two lags to start, so loop begins at t=2
for t in range(2, N1):
    y1[t] = (CONST1
             + PHI1  * y1[t-1]      # AR(1) term
             + PHI2  * y1[t-2]      # AR(2) term
             + eps[t]               # contemporaneous shock
             + THETA1 * eps[t-1])   # MA(1) term

# Wrap in a pandas Series with quarterly date index — makes plots cleaner
dates1     = pd.date_range(start='1970-01-01', periods=N1, freq='QE')
gdp_growth = pd.Series(y1, index=dates1, name='GDP_Growth')

print(f'Simulated {N1} quarters of GDP growth: {dates1[0].date()} → {dates1[-1].date()}')
print(f'Mean: {gdp_growth.mean():.3f}  |  Std: {gdp_growth.std():.3f}  |  '
      f'Min: {gdp_growth.min():.3f}  |  Max: {gdp_growth.max():.3f}')

# Export for R/MATLAB replication
pd.DataFrame({'y': y1}).to_csv(os.path.join(OUT_DIR, 'ex1_data.csv'), index=False)
print('Data saved: ex1_data.csv')

## Stage 1 — Identification

### 1a. Visual inspection

The first question is always: does the series look stationary? We want:
- Time series: mean and variance appear constant over time, no visible trend.
- ACF: spikes decay quickly (stationary) vs. linger near 1 (unit root).
- PACF: used to gauge the AR order — significant spike at lag $k$ then cut-off suggests AR($k$).

In [ ]:
# ── Figure 1a: Time series + ACF + PACF ──────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(12, 9))

# Panel 1: raw time series with mean line
axes[0].plot(gdp_growth, 'steelblue', linewidth=0.9)
axes[0].axhline(gdp_growth.mean(), color='red', linestyle='--',
                label=f'Mean = {gdp_growth.mean():.2f}')
axes[0].set_title('GDP Growth Rate — Simulated Quarterly Data')
axes[0].set_ylabel('Percent')
axes[0].legend()

# Panel 2: ACF — does it decay quickly?
plot_acf(gdp_growth, lags=20, ax=axes[1], alpha=0.05, zero=False)
axes[1].set_title('ACF — Geometric decay suggests AR component')
axes[1].set_xlabel('Lag')

# Panel 3: PACF — where does it cut off?
plot_pacf(gdp_growth, lags=20, ax=axes[2], alpha=0.05, method='ywm', zero=False)
axes[2].set_title('PACF — Significant spikes at lags 1 and 2, then cuts off → AR(2) candidate')
axes[2].set_xlabel('Lag')

fig.suptitle('Example 1 — Stage 1: Identification\n'
             'Visual inspection before formal tests', fontsize=13, fontweight='bold')
fig.tight_layout()
path = os.path.join(OUT_DIR, 'ex1_fig1_identification.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

**What to look for:** The time series should oscillate around a constant mean with no drift — consistent with stationarity ($d=0$). The ACF decays geometrically (not slowly to 1 as for a unit root). The PACF shows significant spikes at lags 1 and 2, then drops inside the confidence bands — this is the fingerprint of an AR(2) process.

### 1b. Formal unit root tests

Visual inspection is suggestive but not conclusive. We use two complementary tests:

- **ADF** (Augmented Dickey-Fuller): $H_0$: unit root. Rejecting means stationary.
- **KPSS**: $H_0$: stationarity. Rejecting means non-stationary.

The **joint inference rule**: conclude stationarity only if ADF rejects *and* KPSS does not reject. When they conflict, the series is ambiguous (near unit root).

In [ ]:
# ── Stage 1b: Unit root tests ─────────────────────────────────────────────────

# ADF test — lag length chosen by AIC (autolag='AIC')
adf = adfuller(gdp_growth, autolag='AIC')
print('ADF Test  (H₀: unit root exists)')
print(f'  Statistic : {adf[0]:.4f}')
print(f'  p-value   : {adf[1]:.4f}')
print(f'  CV 1%/5%/10%: {adf[4]["1%"]:.2f} / {adf[4]["5%"]:.2f} / {adf[4]["10%"]:.2f}')
print(f'  → {"REJECT H₀ — series is stationary" if adf[1] < 0.05 else "FAIL to reject H₀ — unit root possible"}')

print()

# KPSS test — regression='c' tests stationarity around a constant
kpss_res = kpss(gdp_growth, regression='c', nlags='auto')
print('KPSS Test (H₀: series is stationary)')
print(f'  Statistic : {kpss_res[0]:.4f}')
print(f'  p-value   : {kpss_res[1]:.4f}')
print(f'  CV 1%/5%/10%: {kpss_res[3]["1%"]:.2f} / {kpss_res[3]["5%"]:.2f} / {kpss_res[3]["10%"]:.2f}')
print(f'  → {"REJECT H₀ — non-stationary" if kpss_res[1] < 0.05 else "FAIL to reject H₀ — consistent with stationarity"}')

print()
print('Joint inference:')
adf_stat  = adf[1] < 0.05
kpss_stat = kpss_res[1] >= 0.05
if adf_stat and kpss_stat:
    print('  ✓ Both tests agree: STATIONARY → d = 0, no differencing needed')
elif not adf_stat and not kpss_stat:
    print('  ✗ Both tests agree: UNIT ROOT → d = 1, take first differences')
else:
    print('  ⚠ Tests conflict: borderline case — inspect carefully')

## Stage 2 — Estimation

Since the series is stationary ($d=0$), we fit a range of ARMA($p,q$) models by **maximum likelihood** and compare using information criteria:

$$\text{AIC} = -2\ell + 2k, \qquad \text{BIC} = -2\ell + k\ln(T)$$

where $\ell$ is the log-likelihood and $k$ the number of parameters. **BIC penalises complexity more heavily** and tends to select more parsimonious models, which is why it is preferred in macroeconometrics where over-fitting is costly.

In [ ]:
# ── Stage 2: Fit candidate models and compare by AIC/BIC ─────────────────────
ic_results = []

print(f'{"Model":<16} {"Log-Lik":>10} {"AIC":>10} {"BIC":>10}')
print('-' * 50)

for order in CANDIDATE_ORDERS:
    try:
        # ARIMA fit via maximum likelihood
        fitted = ARIMA(gdp_growth, order=order).fit()
        ic_results.append({'order': order, 'aic': fitted.aic,
                            'bic': fitted.bic, 'model': fitted})
        label = f'ARIMA{order}'
        print(f'{label:<16} {fitted.llf:>10.2f} {fitted.aic:>10.2f} {fitted.bic:>10.2f}')
    except Exception as e:
        print(f'ARIMA{order:<10} FAILED: {e}')

print('-' * 50)

# Select the model with lowest BIC (most parsimonious fit)
best = min(ic_results, key=lambda x: x['bic'])
print(f'\n→ Best model by BIC: ARIMA{best["order"]}  '
      f'(true DGP is ARMA(2,1) = ARIMA(2,0,1))')

In [ ]:
# ── IC comparison bar chart ───────────────────────────────────────────────────
# Visualising AIC/BIC differences makes model selection more transparent than a table alone.
labels_ic = [f'ARIMA{r["order"]}' for r in ic_results]
aics = [r['aic'] for r in ic_results]
bics = [r['bic'] for r in ic_results]

# Normalise relative to the minimum (ΔIC) — standard practice in model comparison
delta_aic = np.array(aics) - min(aics)
delta_bic = np.array(bics) - min(bics)

x = np.arange(len(labels_ic))
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, delta, ic_name, color in [
    (axes[0], delta_aic, 'ΔAIC', '#5DADE2'),
    (axes[1], delta_bic, 'ΔBIC', '#E74C6F'),
]:
    bars = ax.bar(x, delta, color=color, edgecolor='white', width=0.6)
    # Highlight the winning model
    bars[np.argmin(delta)].set_edgecolor('black')
    bars[np.argmin(delta)].set_linewidth(2.5)
    ax.set_xticks(x)
    ax.set_xticklabels(labels_ic, rotation=15, ha='right')
    ax.set_ylabel(f'{ic_name} (relative to best)')
    ax.set_title(f'{ic_name} — lower is better')
    ax.axhline(0, color='black', linewidth=0.7)
    # Rule of thumb: ΔBIC > 10 is decisive evidence against a model
    if ic_name == 'ΔBIC':
        ax.axhline(10, color='gray', linestyle=':', linewidth=1.2, label='ΔBIC=10 (decisive)')
        ax.legend()

fig.suptitle('Example 1 — Stage 2: Model Comparison\n'
             'Black border = selected model', fontsize=13, fontweight='bold')
fig.tight_layout()
path = os.path.join(OUT_DIR, 'ex1_fig2_model_comparison.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

**What to look for:** The best model should have ΔBIC = 0. A ΔBIC > 10 relative to the winner is considered decisive evidence against a model (Kass & Raftery, 1995). Models with fewer parameters than the true DGP (e.g. AR(1)) should show clearly worse fit.

## Stage 3 — Diagnostic checking

A correctly specified ARIMA model should leave **white noise residuals** — no remaining autocorrelation, no systematic patterns. If any structure remains, the model is misspecified.

We check with:
1. **Time series plot**: no visible patterns or outliers.
2. **ACF of residuals**: all spikes inside the 95% band.
3. **Histogram + Q-Q plot**: normality of residuals.
4. **Ljung-Box test**: formal $H_0$ of no autocorrelation up to lag $m$. A $p$-value > 0.05 at all lags means the model passes.

In [ ]:
# ── Stage 3: Diagnostic checks on the selected model ─────────────────────────
best_model1 = best['model']
resid1      = best_model1.resid

# Number of parameters used (degrees of freedom correction for Ljung-Box)
p_best, d_best, q_best = best['order']
n_params = p_best + q_best          # df correction in LB test

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Panel (a): Residuals over time — look for outliers or changing variance
axes[0,0].plot(resid1.index, resid1, 'steelblue', linewidth=0.8)
axes[0,0].axhline(0, color='red', linestyle='--', linewidth=1)
axes[0,0].set_title('(a) Residuals over time\n(should look like white noise)')
axes[0,0].set_ylabel('Residuals')

# Panel (b): Histogram + fitted normal — heavy tails would suggest t-distributed errors
axes[0,1].hist(resid1, bins=28, density=True, alpha=0.7,
               color='steelblue', edgecolor='white')
xr = np.linspace(resid1.min(), resid1.max(), 200)
axes[0,1].plot(xr, stats.norm.pdf(xr, resid1.mean(), resid1.std()),
               'r-', linewidth=2, label='Normal fit')
axes[0,1].set_title('(b) Histogram of residuals')
axes[0,1].legend()

# Panel (c): ACF of residuals — all spikes should be inside the band
plot_acf(resid1, lags=20, ax=axes[1,0], alpha=0.05, zero=False)
axes[1,0].set_title('(c) ACF of residuals\n(no spikes outside 95% band = good)')
axes[1,0].set_xlabel('Lag')

# Panel (d): Q-Q plot — points on the diagonal = Gaussian residuals
stats.probplot(resid1, dist='norm', plot=axes[1,1])
axes[1,1].set_title('(d) Q-Q plot — heavy tails appear as S-curve deviations')

fig.suptitle(f'Example 1 — Stage 3: Diagnostics for ARIMA{best["order"]}',
             fontsize=13, fontweight='bold')
fig.tight_layout()
path = os.path.join(OUT_DIR, 'ex1_fig3_diagnostics.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

In [ ]:
# ── Formal tests on residuals ─────────────────────────────────────────────────

# Ljung-Box test — H₀: no autocorrelation up to lag m
# model_df subtracts the number of estimated parameters from the degrees of freedom
lb = acorr_ljungbox(resid1, lags=[5, 10, 15, 20],
                    model_df=n_params, return_df=True)
print(f'Ljung-Box Test on residuals of ARIMA{best["order"]}')
print(f'(H₀: no autocorrelation — want p-value > 0.05 at all lags)')
print()
print(f'{"Lag":>5} {"LB stat":>10} {"p-value":>10} {"Pass?":>8}')
print('-' * 38)
for lag, row in lb.iterrows():
    ok = '✓' if row['lb_pvalue'] > 0.05 else '✗'
    print(f'{int(lag):>5} {row["lb_stat"]:>10.3f} {row["lb_pvalue"]:>10.4f} {ok:>8}')

# Jarque-Bera normality test on residuals
jb_stat, jb_p = stats.jarque_bera(resid1)
print(f'\nJarque-Bera normality test:')
print(f'  Statistic: {jb_stat:.3f}  |  p-value: {jb_p:.4f}')
print(f'  → {"Residuals consistent with normality ✓" if jb_p > 0.05 else "Non-normal residuals — consider t-errors"}')

print(f'\nResidual summary statistics:')
print(f'  Mean     : {resid1.mean():.4f}  (should be ≈ 0)')
print(f'  Std dev  : {resid1.std():.4f}')
print(f'  Skewness : {stats.skew(resid1):.4f}  (should be ≈ 0)')
print(f'  Kurtosis : {stats.kurtosis(resid1):.4f}  (excess, should be ≈ 0)')

**What to look for:** All Ljung-Box $p$-values should exceed 0.05. If some fail, the model needs more AR or MA lags. A Jarque-Bera $p$-value < 0.05 suggests non-Gaussian errors — not a problem for point estimates but affects inference; consider using heteroskedasticity-robust standard errors or a $t$-distribution.

---
# Example 2 — Spurious Regression, Cointegration & ECM
### Engle-Granger Two-Step Procedure
*Textbook reference: Section 1.7–1.8*

## Background

Example 1 assumed the series was stationary. When working with macroeconomic levels (GDP, prices, exchange rates), most series are I(1). This creates two scenarios:

**Scenario A — Spurious regression:** Regressing one I(1) series on another *unrelated* I(1) series produces large $R^2$ and significant $t$-stats purely because both series wander — not because they are related. The residuals will be non-stationary (DW ≈ 0).

**Scenario B — Cointegration:** When two I(1) series share a **common stochastic trend**, their linear combination $z_t = y_t - \alpha - \beta x_t$ is I(0). This is the Engle-Granger definition of cointegration. The long-run relationship is genuine, and an **Error Correction Model (ECM)** governs the short-run dynamics:

$$\Delta y_t = \gamma + \underbrace{\alpha}_{<0} \underbrace{z_{t-1}}_{\text{equilibrium error}} + \delta \Delta x_t + \varepsilon_t$$

The coefficient $\alpha < 0$ ensures the system corrects back to equilibrium whenever $z_{t-1} \neq 0$.

In [ ]:
# 🎛️ EXAMPLE 2 PARAMETERS
N2         = 200    # sample size
# True cointegrating relationship: c = ALPHA_TRUE + BETA_TRUE * y
ALPHA_TRUE = 15.0   # intercept  (try 10 or 20)
BETA_TRUE  = 0.85   # long-run MPC (marginal propensity to consume)
PHI_EC     = 0.6    # AR(1) coefficient of the stationary error around cointegration
SIGMA_EC   = 1.5    # std dev of that error
DRIFT      = 0.5    # drift in income random walk (average income growth per period)

print(f'True long-run relationship: consumption = {ALPHA_TRUE} + {BETA_TRUE}·income + stationary error')

In [ ]:
# ── Part A: Spurious regression — find a visually dramatic seed ───────────────
# We search over seeds to find one where the spurious R² is high,
# making the misleading nature of the result maximally clear.

def find_high_r2_seed(n, target_r2=0.85, max_seeds=2000):
    """Return (seed, R²) for the most dramatic spurious regression found."""
    best_seed, best_r2 = 0, 0
    for seed in range(max_seeds):
        np.random.seed(seed)
        y_rw = np.cumsum(np.random.normal(0, 1, n))    # random walk 1
        x_rw = np.cumsum(np.random.normal(0, 1, n))    # random walk 2 (independent)
        m    = OLS(y_rw, sm.add_constant(x_rw)).fit()
        if m.rsquared > best_r2:
            best_r2 = m.rsquared
            best_seed = seed
            if best_r2 >= target_r2:
                break
    return best_seed, best_r2

spur_seed, spur_r2 = find_high_r2_seed(N2, target_r2=0.85)
print(f'Found seed {spur_seed} → spurious R² = {spur_r2:.3f}')

# Generate the spurious pair with the selected seed
np.random.seed(spur_seed)
y_spur = np.cumsum(np.random.normal(0, 1, N2))    # independent random walk 1
x_spur = np.cumsum(np.random.normal(0, 1, N2))    # independent random walk 2

# Fit spurious OLS regression
spur_model = OLS(y_spur, sm.add_constant(x_spur)).fit()
spur_dw    = sm.stats.durbin_watson(spur_model.resid)
spur_r2    = spur_model.rsquared

print(f'\nSpurious regression: y = {spur_model.params[0]:.2f} + {spur_model.params[1]:.3f}·x')
print(f'  R² = {spur_r2:.3f}  |  t = {spur_model.tvalues[1]:.1f}  |  DW = {spur_dw:.3f}')
print(f'  Granger-Newbold: R² > DW → {"SPURIOUS ⚠" if spur_r2 > spur_dw else "OK"}')

# ADF on spurious residuals — should be NON-STATIONARY
adf_spur = adfuller(spur_model.resid, autolag='AIC')
print(f'  ADF on residuals: stat={adf_spur[0]:.3f}, p={adf_spur[1]:.3f} '
      f'→ {"Stationary (unusual)" if adf_spur[1]<0.05 else "NON-STATIONARY — spurious confirmed"}')

In [ ]:
# ── Figure 2a: Spurious regression illustration ───────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Panel (a): Both random walks over time — visually they seem to co-move
axes[0,0].plot(y_spur, color='steelblue', linewidth=1.1, label='$y_t$ (RW 1)')
axes[0,0].plot(x_spur, color='darkred',   linewidth=1.1, label='$x_t$ (RW 2, INDEPENDENT)')
axes[0,0].axhline(0, color='gray', linestyle='--', linewidth=0.7, alpha=0.6)
axes[0,0].set_title('(a) Two Independent Random Walks\n(no true relationship exists!)')
axes[0,0].legend()

# Panel (b): Scatter plot — OLS finds a significant slope
axes[0,1].scatter(x_spur, y_spur, alpha=0.45, s=20, color='purple', edgecolors='none')
x_rng   = np.linspace(x_spur.min(), x_spur.max(), 100)
y_fit   = spur_model.params[0] + spur_model.params[1] * x_rng
axes[0,1].plot(x_rng, y_fit, color='red', linewidth=2.2,
               label=f'OLS: $\\hat{{\\beta}}$={spur_model.params[1]:.3f}')
axes[0,1].set_xlabel('$x_t$')
axes[0,1].set_ylabel('$y_t$')
axes[0,1].set_title(f'(b) Spurious fit: $R^2$={spur_r2:.3f}, $t$={spur_model.tvalues[1]:.1f}')
axes[0,1].legend()
# Warning text box
axes[0,1].text(0.05, 0.95,
               f'$R^2={spur_r2:.3f}$, $p<0.001$\nYet NO true relationship!',
               transform=axes[0,1].transAxes, fontsize=10,
               verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='lightyellow',
                         edgecolor='orange', alpha=0.9))

# Panel (c): Residuals — should be stationary if regression is valid, but they wander
axes[1,0].plot(spur_model.resid, color='darkgreen', linewidth=0.9)
axes[1,0].axhline(0, color='red', linestyle='--', linewidth=1.4)
axes[1,0].fill_between(range(N2), spur_model.resid, alpha=0.25, color='darkgreen')
axes[1,0].set_title(f'(c) Residuals: DW = {spur_dw:.3f}\n(wandering ≠ white noise)')
axes[1,0].set_ylabel('Residuals')

# Panel (d): ACF of residuals — slow decay is the hallmark of a unit root
plot_acf(spur_model.resid, ax=axes[1,1], lags=30, alpha=0.05, zero=False)
axes[1,1].set_title('(d) ACF of residuals\n(slow linear decay → residuals have unit root)')
axes[1,1].set_xlabel('Lag')

fig.suptitle('Example 2 — Part A: Spurious Regression\n'
             'High R² + significant t + wandering residuals = meaningless result',
             fontsize=13, fontweight='bold')
fig.tight_layout()
path = os.path.join(OUT_DIR, 'ex2_fig1_spurious.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

**What to look for:** High $R^2$ and significant $t$-stat despite no true relationship. The residuals in panel (c) wander persistently — a random walk — not the white noise pattern you would see from a valid regression. The ACF in panel (d) decays so slowly it barely moves — the correlogram fingerprint of a unit root.

## Part B — Cointegration: Engle-Granger two-step

Now we generate a **genuine** long-run relationship (Permanent Income Hypothesis: consumption is proportional to income). Both series are I(1), but their linear combination is I(0) — they are cointegrated.

**Engle-Granger two-step:**
1. Estimate the long-run (cointegrating) regression by OLS: $c_t = \hat{\alpha} + \hat{\beta} y_t + \hat{z}_t$
2. Test $\hat{z}_t$ for stationarity with ADF. Use Engle-Granger critical values (more negative than standard ADF because we estimated $\hat{\beta}$ — an extra estimation step inflates the test).

In [ ]:
# ── Part B: Generate cointegrated consumption-income series ───────────────────
np.random.seed(456)    # separate seed so cointegration example is distinct from spurious

# Income: random walk with drift (average growth per period)
income = 100 + np.cumsum(np.random.normal(DRIFT, 1.0, N2))

# Stationary equilibrium error: AR(1) — represents transitory deviations from PIH
eq_error = np.zeros(N2)
for t in range(1, N2):
    eq_error[t] = PHI_EC * eq_error[t-1] + np.random.normal(0, SIGMA_EC)

# Consumption = long-run relationship + stationary deviation
# The linear combination (consumption - alpha - beta*income) is I(0) by construction
consumption = ALPHA_TRUE + BETA_TRUE * income + eq_error

# ── Step 1: Unit root tests to confirm both series are I(1) ──────────────────
print('Unit root tests in LEVELS (expect FAIL to reject H₀ → I(1)):')
for name, series in [('income', income), ('consumption', consumption)]:
    adf_lev = adfuller(series, autolag='AIC')
    print(f'  {name:14s}: ADF stat={adf_lev[0]:.3f}, p={adf_lev[1]:.3f} '
          f'→ {"I(0) stationary" if adf_lev[1]<0.05 else "I(1) unit root"}')

print('\nUnit root tests on FIRST DIFFERENCES (expect REJECT → I(0)):')
for name, series in [('Δincome', np.diff(income)), ('Δconsumption', np.diff(consumption))]:
    adf_dif = adfuller(series, autolag='AIC')
    print(f'  {name:14s}: ADF stat={adf_dif[0]:.3f}, p={adf_dif[1]:.4f} '
          f'→ {"I(0) stationary ✓" if adf_dif[1]<0.05 else "Still non-stationary"}')

In [ ]:
# ── Step 2: Engle-Granger cointegrating regression ───────────────────────────
# OLS on I(1) levels is valid ONLY if the series are cointegrated.
# If cointegrated, OLS gives a superconsistent estimator of the long-run parameters.
coint_model = OLS(consumption, sm.add_constant(income)).fit()
z_hat       = coint_model.resid    # estimated equilibrium error
dw_coint    = sm.stats.durbin_watson(z_hat)

print('Cointegrating regression: consumption = α + β·income + z')
print(f'  α̂ = {coint_model.params[0]:.4f}  (true = {ALPHA_TRUE})')
print(f'  β̂ = {coint_model.params[1]:.4f}  (true = {BETA_TRUE})')
print(f'  R² = {coint_model.rsquared:.4f}  |  DW = {dw_coint:.4f}')

# ── Step 3: ADF on cointegrating residuals ────────────────────────────────────
# CRITICAL: use Engle-Granger critical values, not standard ADF ones.
# EG CVs for 2 variables: -3.90 (1%), -3.34 (5%), -3.04 (10%)
adf_eg = adfuller(z_hat, autolag='AIC', regression='n')   # regression='n': no constant in ADF (already estimated)
eg_cv5 = -3.34

print(f'\nADF on cointegrating residuals z̃_t:')
print(f'  ADF statistic : {adf_eg[0]:.4f}')
print(f'  Standard p    : {adf_eg[1]:.4f}  (⚠ use EG critical values instead)')
print(f'  EG CV (5%)    : {eg_cv5:.2f}')
print(f'  ADF < EG CV?  : {adf_eg[0]:.3f} < {eg_cv5:.2f} → {"YES — reject no cointegration ✓" if adf_eg[0] < eg_cv5 else "NO — no cointegration"}')

In [ ]:
# ── Figure 2b: Cointegration illustration ────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Panel (a): Both I(1) series — they move together (cointegrated)
axes[0,0].plot(income,      color='steelblue', linewidth=1.1, label='Income $y_t$')
axes[0,0].plot(consumption, color='darkred',   linewidth=1.1, label='Consumption $c_t$')
axes[0,0].set_title('(a) Cointegrated Series\n(both I(1) but sharing a common trend)')
axes[0,0].legend(loc='upper left')

# Panel (b): Cointegrating regression scatter
axes[0,1].scatter(income, consumption, alpha=0.4, s=20, color='purple', edgecolors='none')
inc_rng = np.linspace(income.min(), income.max(), 100)
axes[0,1].plot(inc_rng,
               coint_model.params[0] + coint_model.params[1] * inc_rng,
               color='red', linewidth=2.2,
               label=f'$\\hat{{c}} = {coint_model.params[0]:.1f} + {coint_model.params[1]:.3f}\\cdot y$')
axes[0,1].set_xlabel('Income')
axes[0,1].set_ylabel('Consumption')
axes[0,1].set_title(f'(b) Cointegrating regression: $R^2$={coint_model.rsquared:.3f}')
axes[0,1].legend(loc='upper left')

# Panel (c): Equilibrium error z_hat — must be stationary for cointegration
axes[1,0].plot(z_hat, color='darkgreen', linewidth=0.9)
axes[1,0].axhline(0, color='red', linestyle='--', linewidth=1.4)
axes[1,0].fill_between(range(N2), z_hat, alpha=0.25, color='darkgreen')
axes[1,0].set_title('(c) Equilibrium error $\\hat{z}_t = c_t - \\hat{\\alpha} - \\hat{\\beta}y_t$\n(stationary: mean-reverting)')
axes[1,0].set_ylabel('Equilibrium Error')

# Panel (d): ACF of equilibrium error — decays quickly (stationary)
plot_acf(z_hat, ax=axes[1,1], lags=30, alpha=0.05, zero=False)
axes[1,1].set_title('(d) ACF of $\\hat{z}_t$\n(fast decay confirms stationarity)')
axes[1,1].set_xlabel('Lag')

fig.suptitle('Example 2 — Part B: Cointegration\n'
             'Stationary equilibrium error = genuine long-run relationship',
             fontsize=13, fontweight='bold')
fig.tight_layout()
path = os.path.join(OUT_DIR, 'ex2_fig2_cointegration.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

**What to look for:** Compare panels (c) and (d) here with those in Part A. The equilibrium error $\hat{z}_t$ now oscillates around zero without wandering — it is stationary. The ACF decays quickly (geometric) rather than showing the slow near-unity decay seen in the spurious case.

## Part C — Error Correction Model

Cointegration has a profound economic implication (Granger Representation Theorem): if two series are cointegrated, there *must* exist an ECM governing their short-run dynamics. The ECM separates:
- **Long-run effect** ($\hat{\beta}$): how much consumption changes permanently with income.
- **Short-run effect** ($\delta$): the immediate response of $\Delta c_t$ to $\Delta y_t$.
- **Speed of adjustment** ($\alpha < 0$): how fast the system returns to equilibrium after a shock.

In [ ]:
# ── Part C: Estimate the Error Correction Model ───────────────────────────────

# ECM variables:
dc     = np.diff(consumption)    # Δc_t: change in consumption
dy     = np.diff(income)         # Δy_t: change in income
z_lag  = z_hat[:-1]              # z_{t-1}: lagged equilibrium error (error correction term)

# OLS regression: Δc_t = γ + α·z_{t-1} + δ·Δy_t + ε_t
ecm_df    = pd.DataFrame({'dc': dc, 'dy': dy, 'z_lag': z_lag})
ecm_model = OLS(ecm_df['dc'],
                sm.add_constant(ecm_df[['z_lag', 'dy']])).fit()

alpha_ec  = ecm_model.params['z_lag']    # error correction coefficient
delta_ec  = ecm_model.params['dy']       # short-run income effect
beta_lr   = coint_model.params[1]        # long-run income effect (from cointegrating reg)

print('ECM: Δc_t = γ + α·z_{t-1} + δ·Δy_t + ε_t')
print(f'{"Parameter":<25} {"Estimate":>10} {"Std Err":>10} {"t-stat":>10}')
print('-' * 58)
for name, param_key in [("Constant (γ)", "const"),
                         ("Error correction (α)", "z_lag"),
                         ("Δ Income (δ)", "dy")]:
    print(f'{name:<25} {ecm_model.params[param_key]:>10.4f} '
          f'{ecm_model.bse[param_key]:>10.4f} '
          f'{ecm_model.tvalues[param_key]:>10.2f}')
print('-' * 58)
print(f'R² = {ecm_model.rsquared:.4f}  |  DW = {sm.stats.durbin_watson(ecm_model.resid):.4f}')

# Economic interpretation
print()
if alpha_ec < 0:
    half_life_ec = np.log(0.5) / np.log(1 + alpha_ec)   # periods for 50% of shock to dissipate
    print(f'✓ α = {alpha_ec:.4f} < 0 → system corrects toward equilibrium')
    print(f'  Speed of adjustment : {abs(alpha_ec)*100:.1f}% of disequilibrium corrected per period')
    print(f'  Half-life of shocks : {half_life_ec:.1f} periods')
else:
    print(f'✗ α = {alpha_ec:.4f} > 0 → explosive dynamics!')

print(f'\nShort-run vs long-run:')
print(f'  Immediate effect of Δy on Δc (δ) : {delta_ec:.4f}')
print(f'  Long-run effect of y on c (β)    : {beta_lr:.4f}')
print(f'  → $1 rise in income: ${delta_ec:.2f} immediate, adjusts to ${beta_lr:.2f} in long run')

In [ ]:
# ── Figure 2c: ECM diagnostics ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Panel (a): Equilibrium error with ±1.5σ bands to show mean reversion
z_sd = z_hat.std()
axes[0,0].plot(z_hat, color='steelblue', linewidth=0.9)
axes[0,0].axhline(0,      color='red',  linestyle='--', linewidth=1.5)
axes[0,0].axhline( 1.5*z_sd, color='orange', linestyle=':', linewidth=1.2, label='±1.5σ')
axes[0,0].axhline(-1.5*z_sd, color='orange', linestyle=':', linewidth=1.2)
axes[0,0].fill_between(range(N2), z_hat, alpha=0.25, color='steelblue')
axes[0,0].set_title('(a) Equilibrium Error $z_t$\n(mean-reversion = error correction at work)')
axes[0,0].set_ylabel('$z_t$')
axes[0,0].legend()

# Panel (b): Actual vs fitted Δc — how well does the ECM track short-run changes?
axes[0,1].plot(dc, color='steelblue', linewidth=0.8, alpha=0.8, label='Actual $\\Delta c_t$')
axes[0,1].plot(ecm_model.fittedvalues.values, color='red', linewidth=1.3, label='ECM fitted')
axes[0,1].set_title(f'(b) Actual vs Fitted $\\Delta c_t$: $R^2$={ecm_model.rsquared:.3f}')
axes[0,1].set_ylabel('$\\Delta c_t$')
axes[0,1].legend()

# Panel (c): ECM residuals — should be white noise
axes[1,0].plot(ecm_model.resid.values, color='darkgreen', linewidth=0.9)
axes[1,0].axhline(0, color='red', linestyle='--', linewidth=1.4)
axes[1,0].set_title(f'(c) ECM Residuals (DW={sm.stats.durbin_watson(ecm_model.resid):.2f})')

# Panel (d): Error correction mechanism — scatter of z_{t-1} vs Δc_t
# The negative slope IS the error correction: when above equilibrium, consumption falls
axes[1,1].scatter(z_lag, dc, alpha=0.4, s=22, color='purple', edgecolors='none')
axes[1,1].axhline(0, color='gray', linestyle='--', alpha=0.7)
axes[1,1].axvline(0, color='gray', linestyle='--', alpha=0.7)
z_rng  = np.linspace(z_lag.min(), z_lag.max(), 100)
ec_fit = alpha_ec * z_rng + ecm_model.params['const']
axes[1,1].plot(z_rng, ec_fit, color='red', linewidth=2.2,
               label=f'Slope = α = {alpha_ec:.3f}')
axes[1,1].set_xlabel('Lagged equilibrium error $z_{t-1}$')
axes[1,1].set_ylabel('$\\Delta c_t$')
axes[1,1].set_title('(d) Error Correction Mechanism\n(negative slope = pull back to equilibrium)')
axes[1,1].legend()
# Annotate the economic intuition
axes[1,1].text(0.62, 0.93, 'Above equilibrium\n→ Δc < 0',
               transform=axes[1,1].transAxes, fontsize=9, color='darkred',
               verticalalignment='top')
axes[1,1].text(0.02, 0.12, 'Below equilibrium\n→ Δc > 0',
               transform=axes[1,1].transAxes, fontsize=9, color='darkgreen')

fig.suptitle('Example 2 — Part C: Error Correction Model\n'
             f'α={alpha_ec:.3f} ({abs(alpha_ec)*100:.0f}% correction/period), '
             f'half-life={half_life_ec:.1f} periods',
             fontsize=13, fontweight='bold')
fig.tight_layout()
path = os.path.join(OUT_DIR, 'ex2_fig3_ecm.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

In [ ]:
# ── Figure 2d: Spurious vs cointegrated — the key diagnostic comparison ────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, resid, r2, dw, title_label, fill_color, note, note_color in [
    (axes[0], spur_model.resid, spur_r2, spur_dw,
     'SPURIOUS regression', 'crimson',
     'Non-stationary!\nWanders without\nreturning to zero', 'lightyellow'),
    (axes[1], z_hat, coint_model.rsquared, dw_coint,
     'COINTEGRATING regression', 'steelblue',
     'Stationary!\nMean-reverting\naround zero', 'lightgreen'),
]:
    ax.plot(resid, color=fill_color, linewidth=0.9)
    ax.axhline(0, color='black', linestyle='--', linewidth=1)
    ax.fill_between(range(len(resid)), resid, alpha=0.25, color=fill_color)
    ax.set_title(f'{title_label} residuals\n$R^2$={r2:.3f}, DW={dw:.3f}')
    ax.set_ylabel('Residuals')
    ax.set_xlabel('Time')
    ax.text(0.04, 0.95, note, transform=ax.transAxes,
            fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor=note_color, alpha=0.9))

fig.suptitle('Example 2 — Residuals: Spurious vs Cointegrated\n'
             'The key diagnostic is residual stationarity, NOT R²',
             fontsize=13, fontweight='bold')
fig.tight_layout()
path = os.path.join(OUT_DIR, 'ex2_fig4_comparison.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

# Save data for R/MATLAB
pd.DataFrame({'income': income, 'consumption': consumption}).to_csv(
    os.path.join(OUT_DIR, 'ex2_data.csv'), index=False)
print('Data saved: ex2_data.csv')

**What to look for in Figure 2d:** This is the core diagnostic that separates spurious from valid regressions. Both have high $R^2$, but only the cointegrated regression has stationary residuals. In applied work you cannot distinguish them by looking at $R^2$ or $t$-stats alone — you *must* test the residuals.

---
# Example 3 — GARCH Volatility Modelling
### ARCH-LM Test · GARCH(1,1) · Asymmetric Models
*Textbook reference: Section 1.9*

## Background

Financial returns often exhibit **volatility clustering**: large movements tend to be followed by large movements (of either sign), and calm periods are followed by calm periods. ARMA models cannot capture this because they assume a constant error variance ($\sigma^2$). The **GARCH** family instead models the conditional variance as:

$$\sigma^2_t = \omega + \alpha \varepsilon^2_{t-1} + \beta \sigma^2_{t-1}$$

- $\omega > 0$: baseline (unconditional) variance floor.
- $\alpha \geq 0$: **ARCH effect** — how much yesterday's squared shock affects today's variance.
- $\beta \geq 0$: **GARCH effect** — how much yesterday's variance persists.
- $\alpha + \beta < 1$ ensures stationarity; the closer to 1, the higher the **persistence**.

**Unconditional variance**: $\sigma^2 = \omega / (1 - \alpha - \beta)$.
**Half-life**: $\ln(0.5) / \ln(\alpha + \beta)$ periods for a volatility shock to decay by 50%.

In [ ]:
# 🎛️ EXAMPLE 3 PARAMETERS
N3         = 1000   # sample size (daily observations — GARCH needs long samples)
SEED3      = 456
# True GARCH(1,1) parameters
OMEGA_TRUE = 0.05   # variance intercept
ALPHA_TRUE3 = 0.10  # ARCH coefficient (reaction to shocks)
BETA_TRUE3  = 0.85  # GARCH coefficient (persistence)
# Note: alpha + beta = 0.95 → high persistence, half-life ≈ 13 days

persist = ALPHA_TRUE3 + BETA_TRUE3
uncond_var_true = OMEGA_TRUE / (1 - persist)
hl_true = np.log(0.5) / np.log(persist)
print(f'True GARCH(1,1) parameters:')
print(f'  ω={OMEGA_TRUE}, α={ALPHA_TRUE3}, β={BETA_TRUE3}')
print(f'  Persistence α+β = {persist}  |  Unconditional var = {uncond_var_true:.4f}')
print(f'  Half-life = {hl_true:.1f} periods')

In [ ]:
# ── Step 0: Simulate GARCH(1,1) data ─────────────────────────────────────────
# The simulation makes the DGP explicit — in practice you load return data from a source.
np.random.seed(SEED3)

returns3 = np.zeros(N3)
sigma2_true = np.zeros(N3)
# Initialise variance at its unconditional (long-run) value
sigma2_true[0] = uncond_var_true

for t in range(1, N3):
    # Update conditional variance: GARCH recursion
    sigma2_true[t] = (OMEGA_TRUE
                      + ALPHA_TRUE3 * returns3[t-1]**2    # ARCH term
                      + BETA_TRUE3  * sigma2_true[t-1])   # GARCH term
    # Return is standard normal scaled by current volatility
    returns3[t] = np.sqrt(sigma2_true[t]) * np.random.standard_normal()

dates3 = pd.date_range(start='2020-01-01', periods=N3, freq='D')
ret_s  = pd.Series(returns3, index=dates3, name='returns')

print(f'Simulated {N3} daily returns')
print(f'  Mean={ret_s.mean():.4f}  Std={ret_s.std():.4f}  '
      f'Skew={stats.skew(ret_s):.4f}  KurtEx={stats.kurtosis(ret_s):.4f}')
print(f'  Excess kurtosis = {stats.kurtosis(ret_s):.2f} '
      f'(>0 = fat tails, expected under GARCH)')

pd.DataFrame({'returns': returns3, 'sigma2': sigma2_true}).to_csv(
    os.path.join(OUT_DIR, 'ex3_data.csv'), index=False)
print('Data saved: ex3_data.csv')

In [ ]:
# ── Figure 3a: Data overview — volatility clustering + fat tails ──────────────
fig, axes = plt.subplots(3, 1, figsize=(13, 10))

# Panel (a): Returns — calm and turbulent periods cluster together
axes[0].plot(dates3, returns3, 'steelblue', linewidth=0.5)
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].set_title('(a) Simulated Stock Returns — Volatility Clustering')
axes[0].set_ylabel('Returns')

# Panel (b): Squared returns vs true conditional variance
# Squared returns are a noisy proxy for true volatility; GARCH smooths them
axes[1].plot(dates3, returns3**2, color='lightcoral', linewidth=0.4,
             alpha=0.7, label='Squared returns (noisy proxy)')
axes[1].plot(dates3, sigma2_true, color='steelblue', linewidth=1.2,
             label='True $\\sigma^2_t$ (GARCH)')
axes[1].set_title('(b) Volatility: Squared Returns vs True Conditional Variance')
axes[1].set_ylabel('Variance')
axes[1].legend()

# Panel (c): Return distribution vs normal — fat tails are visible
axes[2].hist(returns3, bins=60, density=True, alpha=0.7,
             color='steelblue', edgecolor='white', label='Empirical')
xr3 = np.linspace(returns3.min(), returns3.max(), 300)
axes[2].plot(xr3,
             stats.norm.pdf(xr3, returns3.mean(), returns3.std()),
             'r-', linewidth=2.2, label='Normal fit')
axes[2].set_title('(c) Return Distribution — Fat Tails vs Normal\n'
                  f'(excess kurtosis = {stats.kurtosis(returns3):.2f})')
axes[2].set_xlabel('Returns')
axes[2].legend()

fig.suptitle('Example 3 — Data Overview\n'
             'GARCH features: clustering, fat tails, near-zero mean',
             fontsize=13, fontweight='bold')
fig.tight_layout()
path = os.path.join(OUT_DIR, 'ex3_fig1_data.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

**What to look for:** Panel (a) shows the clustering — narrow, then wide, then narrow again. Panel (b) shows that squared returns are a very noisy volatility proxy; the true $\sigma^2_t$ (blue line) follows the envelope of those spikes much more smoothly. Panel (c) shows the fat tails: the empirical distribution has a sharper peak and heavier tails than the fitted normal.

## Step 1 — ARCH-LM test

Before fitting a GARCH model, we confirm that ARCH effects are actually present. The **ARCH-LM test** regresses $\varepsilon_t^2$ on its own lags and tests whether those lags are jointly significant. Under $H_0$ (no ARCH), the $LM$ statistic follows $\chi^2(m)$.

In [ ]:
# ── Step 1: ARCH-LM test ──────────────────────────────────────────────────────
# If we cannot reject H₀ at any lag, there are no ARCH effects and GARCH is unnecessary.
print('ARCH-LM Test  (H₀: no ARCH effects — conditional variance is constant)')
print(f'{"Lags":>6} {"LM stat":>10} {"p-value":>10} {"Verdict":>20}')
print('-' * 52)

for lags in [1, 5, 10, 20]:
    lm_stat, lm_p, _, _ = het_arch(returns3, nlags=lags)   # returns LM stat and p-value
    verdict = 'ARCH effects ✓' if lm_p < 0.05 else 'No ARCH effects'
    stars   = '***' if lm_p < 0.001 else ('**' if lm_p < 0.01 else ('*' if lm_p < 0.05 else ''))
    print(f'{lags:>6} {lm_stat:>10.3f} {lm_p:>10.6f}{stars:>4}  {verdict:>20}')

print('\n→ Highly significant ARCH effects at all lags: GARCH modelling justified')

## Step 2 — GARCH(1,1) estimation and model comparison

We estimate three models:
- **GARCH(1,1)**: symmetric — positive and negative shocks of the same size have identical effects on future variance.
- **GJR-GARCH**: adds $\gamma \varepsilon^2_{t-1} \mathbb{1}[\varepsilon_{t-1}<0]$ — negative shocks have an *extra* effect (leverage effect). Common in equity markets.
- **EGARCH**: models $\ln(\sigma^2_t)$ — automatically ensures positivity; can capture asymmetry.

All estimated by **maximum likelihood** via the `arch` library.

In [ ]:
# ── Step 2: Estimate GARCH, GJR-GARCH, EGARCH ────────────────────────────────

# GARCH(1,1): symmetric baseline
res_garch  = arch_model(ret_s, vol='Garch', p=1, q=1,
                        mean='Constant').fit(disp='off')

# GJR-GARCH(1,1,1): o=1 adds the asymmetry (leverage) term
res_gjr    = arch_model(ret_s, vol='Garch', p=1, o=1, q=1,
                        mean='Constant').fit(disp='off')

# EGARCH(1,1): models log-variance — no positivity constraint needed
res_egarch = arch_model(ret_s, vol='EGARCH', p=1, q=1,
                        mean='Constant').fit(disp='off')

# ── Model comparison table ────────────────────────────────────────────────────
print(f'{"Model":<22} {"Log-Lik":>10} {"AIC":>10} {"BIC":>10}')
print('-' * 55)
for name, res in [('GARCH(1,1)', res_garch),
                  ('GJR-GARCH(1,1,1)', res_gjr),
                  ('EGARCH(1,1)', res_egarch)]:
    print(f'{name:<22} {res.loglikelihood:>10.2f} {res.aic:>10.2f} {res.bic:>10.2f}')
print('-' * 55)

# Identify best by BIC
model_dict = {'GARCH(1,1)': res_garch, 'GJR-GARCH': res_gjr, 'EGARCH': res_egarch}
best_name3 = min(model_dict, key=lambda k: model_dict[k].bic)
print(f'\nBest model by BIC: {best_name3}')

# ── GARCH(1,1) parameter interpretation ──────────────────────────────────────
omega_hat3 = res_garch.params['omega']
alpha_hat3 = res_garch.params['alpha[1]']
beta_hat3  = res_garch.params['beta[1]']
persist_hat = alpha_hat3 + beta_hat3
uncond_var_hat = omega_hat3 / (1 - persist_hat)
hl_hat = np.log(0.5) / np.log(persist_hat)

print(f'\nGARCH(1,1) parameter estimates vs true values:')
print(f'  {"Parameter":<12} {"True":>8} {"Estimated":>10}')
print(f'  {"ω (omega)":<12} {OMEGA_TRUE:>8.4f} {omega_hat3:>10.4f}')
print(f'  {"α (alpha)":<12} {ALPHA_TRUE3:>8.4f} {alpha_hat3:>10.4f}')
print(f'  {"β (beta)":<12} {BETA_TRUE3:>8.4f} {beta_hat3:>10.4f}')
print(f'  {"α + β":<12} {persist:>8.4f} {persist_hat:>10.4f}')
print(f'\nImplied quantities:')
print(f'  Unconditional variance : {uncond_var_hat:.4f}  (true: {uncond_var_true:.4f})')
print(f'  Half-life of shocks    : {hl_hat:.1f} periods  (true: {hl_true:.1f})')

In [ ]:
# ── Figure 3b: GARCH estimation results ──────────────────────────────────────
cond_vol_hat = res_garch.conditional_volatility   # estimated σ_t (not σ²_t)
std_resid3   = res_garch.std_resid                # standardised residuals ε_t/σ_t

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Panel (a): Estimated vs true conditional volatility
axes[0,0].plot(dates3, cond_vol_hat, 'steelblue', linewidth=0.9, label='Estimated $\\hat{\\sigma}_t$')
axes[0,0].plot(dates3, np.sqrt(sigma2_true), 'r--', linewidth=0.9, alpha=0.7, label='True $\\sigma_t$')
axes[0,0].set_title('(a) Estimated vs True Conditional Volatility $\\sigma_t$')
axes[0,0].set_ylabel('Volatility $\\sigma_t$')
axes[0,0].legend()

# Panel (b): Returns with ±2σ bands — these should contain ≈ 95% of observations
axes[0,1].plot(dates3, returns3, 'gray', linewidth=0.35, alpha=0.8, label='Returns')
axes[0,1].plot(dates3,  2 * cond_vol_hat, 'r-', linewidth=1.0, label='$\\pm 2\\hat{\\sigma}_t$ (95% band)')
axes[0,1].plot(dates3, -2 * cond_vol_hat, 'r-', linewidth=1.0)
axes[0,1].fill_between(dates3, -2*cond_vol_hat, 2*cond_vol_hat, alpha=0.1, color='red')
axes[0,1].set_title('(b) Returns with Dynamic Volatility Bands')
axes[0,1].legend()

# Panel (c): Standardised residuals z_t = ε_t/σ_t — should look like WN(0,1)
axes[1,0].plot(dates3, std_resid3, 'steelblue', linewidth=0.5)
axes[1,0].axhline( 0, color='black', linewidth=0.6)
axes[1,0].axhline( 2, color='red', linestyle='--', linewidth=0.9)
axes[1,0].axhline(-2, color='red', linestyle='--', linewidth=0.9, label='$\\pm 2$ bounds')
axes[1,0].set_title('(c) Standardised Residuals $z_t = \\varepsilon_t/\\hat{\\sigma}_t$\n'
                     '(should look like WN(0,1) if model is correct)')
axes[1,0].set_ylabel('$z_t$')
axes[1,0].legend()

# Panel (d): Q-Q of standardised residuals — should be on the diagonal if Gaussian
stats.probplot(std_resid3, dist='norm', plot=axes[1,1])
axes[1,1].set_title('(d) Q-Q Plot of Standardised Residuals\n'
                     '(S-shaped deviations → t-distribution may fit better)')

fig.suptitle('Example 3 — GARCH(1,1) Estimation Results\n'
             f'α={alpha_hat3:.4f}, β={beta_hat3:.4f}, '
             f'persistence={persist_hat:.4f}, half-life={hl_hat:.1f} days',
             fontsize=13, fontweight='bold')
fig.tight_layout()
path = os.path.join(OUT_DIR, 'ex3_fig2_results.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

## Step 3 — Diagnostic checking

After fitting GARCH, we check the **standardised residuals** $z_t = \varepsilon_t / \hat{\sigma}_t$. If the model is correct:
- $z_t$ should be i.i.d. with mean 0 and variance 1.
- The squared standardised residuals $z_t^2$ should show **no remaining ARCH effects**.

We use the ARCH-LM test and Ljung-Box on $z_t^2$ to confirm this.

In [ ]:
# ── Step 3: Diagnostics on standardised residuals ─────────────────────────────
print('Standardised residuals summary (should be close to standard normal):')
print(f'  Mean     : {std_resid3.mean():.4f}  (target: 0)')
print(f'  Std dev  : {std_resid3.std():.4f}  (target: 1)')
print(f'  Skewness : {stats.skew(std_resid3):.4f}')
print(f'  Kurtosis : {stats.kurtosis(std_resid3):.4f}  (excess; target: 0 for normal)')

print('\nARCH-LM test on standardised residuals z_t:')
print('(H₀: no remaining ARCH effects — want p > 0.05)')
print(f'{"Lags":>6} {"LM stat":>10} {"p-value":>10} {"Verdict":>25}')
print('-' * 56)
for lags in [5, 10, 20]:
    lm_stat, lm_p, _, _ = het_arch(std_resid3, nlags=lags)
    verdict = 'No ARCH effects ✓' if lm_p > 0.05 else 'ARCH effects remain ✗'
    print(f'{lags:>6} {lm_stat:>10.3f} {lm_p:>10.4f}  {verdict}')

print('\nLjung-Box test on squared standardised residuals z_t²:')
lb3 = acorr_ljungbox(std_resid3**2, lags=[10, 20], return_df=True)
print(f'{"Lag":>5} {"LB stat":>10} {"p-value":>10}')
print('-' * 30)
for lag, row in lb3.iterrows():
    ok = '✓' if row['lb_pvalue'] > 0.05 else '✗'
    print(f'{int(lag):>5} {row["lb_stat"]:>10.3f} {row["lb_pvalue"]:>10.4f} {ok}')

In [ ]:
# ── Figure 3c: Persistence and half-life sensitivity ─────────────────────────
# This figure shows how the choice of α + β critically affects the economic
# interpretation: a small increase in persistence dramatically extends the half-life.

persist_grid = np.linspace(0.70, 0.999, 300)     # range of persistence values
hl_grid      = np.log(0.5) / np.log(persist_grid) # corresponding half-lives

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel (a): Half-life as a function of persistence
axes[0].plot(persist_grid, hl_grid, 'steelblue', linewidth=2)
# Mark estimated and true values
for label, pers, hl, col in [
    (f'True (={persist:.2f})',        persist,     hl_true, 'green'),
    (f'Estimated (={persist_hat:.2f})', persist_hat, hl_hat,  'red'),
]:
    axes[0].axvline(pers, color=col, linestyle='--', linewidth=1.4)
    axes[0].axhline(hl,   color=col, linestyle='--', linewidth=1.4)
    axes[0].scatter([pers], [hl], color=col, s=70, zorder=5, label=label)
axes[0].set_xlabel('Persistence $\\alpha + \\beta$')
axes[0].set_ylabel('Half-life (periods)')
axes[0].set_title('(a) Volatility Half-Life vs Persistence\n'
                  'Small Δ(α+β) near 1 → large change in half-life')
axes[0].legend(fontsize=9)
axes[0].set_ylim(0, 60)

# Panel (b): Impulse response of a volatility shock
# Show how a spike in volatility (at t=0) decays under different persistence values
horizon = 50
t_ir    = np.arange(horizon)
for pers_ir, col_ir, lab_ir in [
    (0.80, '#2E7D32', '$\\alpha+\\beta=0.80$ (low persistence)'),
    (0.90, '#F57F17', '$\\alpha+\\beta=0.90$'),
    (0.95, '#E74C6F', '$\\alpha+\\beta=0.95$ (this model)'),
    (0.99, '#1565C0', '$\\alpha+\\beta=0.99$ (near IGARCH)'),
]:
    # Impulse response: a shock of size 1 at t=0, then GARCH decay
    ir = pers_ir ** t_ir    # fraction of initial shock remaining at period t
    axes[1].plot(t_ir, ir, color=col_ir, linewidth=2.0, label=lab_ir)

axes[1].axhline(0.5, color='gray', linestyle=':', linewidth=1.2, label='50% decay (half-life)')
axes[1].set_xlabel('Periods after shock')
axes[1].set_ylabel('Fraction of shock remaining')
axes[1].set_title('(b) Impulse Response: Volatility Shock Decay\n'
                  'High persistence → slow mean reversion')
axes[1].legend(fontsize=8.5)
axes[1].set_ylim(0, 1.05)

fig.suptitle('Example 3 — Volatility Persistence and Half-Life',
             fontsize=13, fontweight='bold')
fig.tight_layout()
path = os.path.join(OUT_DIR, 'ex3_fig3_persistence.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

**What to look for:** Panel (a) shows the nonlinear relationship between persistence and half-life — near $\alpha+\beta=1$ (IGARCH), the half-life becomes infinite (the shock never dies). Panel (b) shows the impulse response under four scenarios: at $\alpha+\beta=0.95$ (this model), it takes about 13 periods to halve; at 0.99, it takes over 60.

---
## Summary — All examples

| Example | Key result | Main diagnostic |
|---|---|---|
| **1 — ARIMA** | Box-Jenkins recovers true ARMA(2,1) | Ljung-Box on residuals: all $p > 0.05$ |
| **2 — Cointegration/ECM** | Spurious R²=high ≠ genuine; ECM adjusts at α% per period | ADF on residuals; Granger-Newbold rule |
| **3 — GARCH** | Persistence near 1 → slow volatility mean reversion | ARCH-LM and LB on standardised residuals |

## Exercises

**Exercise 1.1 (ARIMA):** Change `PHI2` to `0.2` (so both AR roots are the same sign). Does the PACF pattern change? Does BIC still select the correct model?

**Exercise 1.2 (ARIMA):** Add a seasonal component to the simulated GDP growth by adding `0.4*y[t-4]` to the DGP. Does the ACF reveal the seasonal pattern? What model does Box-Jenkins now suggest?

**Exercise 2.1 (Cointegration):** Change `BETA_TRUE` from 0.85 to 1.05. Does the Engle-Granger test still detect cointegration? Try increasing `N2` to 500 — does power improve?

**Exercise 2.2 (ECM):** Change `PHI_EC` (the AR coefficient of the equilibrium error) to 0.9. What happens to the speed of adjustment $\hat{\alpha}$ and the ECM half-life?

**Exercise 3.1 (GARCH):** Set `ALPHA_TRUE3 = 0.20` and `BETA_TRUE3 = 0.75` (same persistence, different mix). Does the GARCH(1,1) still recover the parameters well? How does the half-life change?

**Exercise 3.2 (GARCH):** Generate data from a GJR-GARCH (add a leverage effect) and estimate both GARCH(1,1) and GJR-GARCH. Does BIC correctly identify the asymmetric model?

---
*Macroeconometrics* | Alessia Paccagnini  
Companion code — Chapter 1 Applied Examples | Last updated: March 2026